# 优化器归档与恢复校验：权重保留，状态按指纹验证

目标：把此前已完成的单次优化器归档整理成完整证据链：本地清单→公开不可变版本→服务端内容验证→原文件清理记录→恢复后字节检查。原模型权重保留；本篇不会再次上传、删除、恢复大文件或触碰任何凭据。

案例来自第7500步的约10.90十进制吉字节优化器状态。优化器状态用于恢复训练；只有模型权重通常只足够推理。


In [1]:
from pathlib import Path
import json,copy,hashlib,tempfile,os,sys
import numpy as np
from IPython.display import display,Markdown
TOPIC=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/"code/amd_benchmarks").is_dir())
EVIDENCE=TOPIC/"notebooks/results"
print("只复算已有记录和小型故障夹具，不重跑训练、评估或空间清理。")


只复算已有记录和小型故障夹具，不重跑训练、评估或空间清理。


## 1. 核对三个独立记录

源清单给出文件大小和内容散列，归档状态给出公开仓库及不可变版本，清理后记录给出原优化器是否删除和模型是否保留。三个文件必须指向同一产物。


In [2]:
root=EVIDENCE/"diagnostics"
manifest=json.loads((root/"optimizer-archive-manifest.json").read_text())
archive=json.loads((root/"optimizer-archive-state.json").read_text())
cleanup=json.loads((root/"optimizer-cleanup-verification.json").read_text())
assert archive["status"]=="verified" and archive["models_deleted"] is False
assert archive["commit"]==cleanup["commit"]
source=manifest["files"][0];remote=archive["verified_files"][0]
assert source["sha256"]==remote["sha256"] and source["bytes"]==remote["bytes"]
assert remote["verified_by"]=="lfs_sha256"
assert cleanup["pruned_bytes"]==source["bytes"]
assert cleanup["retained_model_weights"]
print({"不可变版本":archive["commit"],"优化器字节数":source["bytes"],"源内容指纹":source["sha256"],"保留的模型字节数":cleanup["retained_model_weights"][0]["bytes"]})
print(archive["url"])


{'不可变版本': 'd9a0339cbf2d19f67809c1e8c575f942a99718d5', '优化器字节数': 10896059452, '源内容指纹': 'ec3c485298a9cb5d1a2d09b80520db9320361b02758f2c8b2ccd5da892bbbf72', '保留的模型字节数': 7536023136}
https://huggingface.co/Datawhale/every-embodied-amd-training-archive-20261027/tree/d9a0339cbf2d19f67809c1e8c575f942a99718d5/training-states/cup/pi0-clean40-s7500


## 2. 展开恢复后内容验证函数

分块读取以避免将大文件整体载入内存。先检查大小，再核对完整内容散列；文件存在或下载命令退出成功都不足以通过该检查。下面函数不执行删除。


In [3]:
def verify_restored_file(path,expected):
    path=Path(path)
    if path.stat().st_size!=expected["bytes"]: raise ValueError("Restored file size mismatch")
    digest=hashlib.sha256()
    with path.open("rb") as handle:
        while block:=handle.read(8*1024*1024): digest.update(block)
    if digest.hexdigest()!=expected["sha256"]: raise ValueError("Restored file content mismatch")
    return {"bytes":path.stat().st_size,"sha256":digest.hexdigest(),"verified":True}


## 3. 故障复现：等大小损坏仍应被拒绝

用几字节临时文件演示，既不下载优化器，也不改原产物。损坏前后大小相同，只有内容散列能识别。


In [4]:
with tempfile.TemporaryDirectory() as temp:
    path=Path(temp)/"fixture.bin";good=b"optimizer-fixture-v1"
    expected={"bytes":len(good),"sha256":hashlib.sha256(good).hexdigest()}
    path.write_bytes(good);assert verify_restored_file(path,expected)["verified"]
    path.write_bytes(b"X"+good[1:])
    try: verify_restored_file(path,expected)
    except ValueError as exc: print("预期拒绝：",exc)
    else: raise AssertionError("Equal-size corruption must fail")
    path.write_bytes(good);print("恢复正确内容后的验证：",verify_restored_file(path,expected))


预期拒绝： Restored file content mismatch
恢复正确内容后的验证： {'bytes': 20, 'sha256': 'f00289b6d8147308144fa8c4073936d8a48cb7adac26b6b0fb387db2ae8ec3fd', 'verified': True}


## 4. 按需恢复的完整命令与验证顺序

若以后确需恢复此大文件，先确认外置盘容量，再在终端用专用下载入口和既有解释器执行打印的命令。使用不可变版本而非浮动分支；下载缓存与目标都放外置盘。该单元仅打印命令，不执行下载。下载后再显式设置本地文件路径进行完整内容检查。


In [5]:
import shlex
restore_root=Path("/data/Data14TB/01Proj/every-embodied-amd/restored-training-state")
script="from huggingface_hub import hf_hub_download; print(hf_hub_download(repo_id="+repr(archive["repo_id"])+", repo_type='model', revision="+repr(archive["commit"])+", filename="+repr(remote["file"])+", cache_dir='/data/Data14TB/cache/huggingface', local_dir="+repr(str(restore_root))+"))"
command=["/home/ubuntu/.local/bin/lab-download","--download-exec",sys.executable,"-c",script]
print(shlex.join(command))
if os.environ.get("AMD_VERIFY_RESTORED_OPTIMIZER", "0")=="1":
    restored_path=Path(os.environ["AMD_RESTORED_OPTIMIZER_PATH"])
    print(verify_restored_file(restored_path,source))
else: print("大文件恢复与完整校验关闭；本轮只核对历史归档证据和损坏夹具。")


/home/ubuntu/.local/bin/lab-download --download-exec /data/Data14TB/envs/every-embodied-pnp-cu128/bin/python -c 'from huggingface_hub import hf_hub_download; print(hf_hub_download(repo_id='"'"'Datawhale/every-embodied-amd-training-archive-20261027'"'"', repo_type='"'"'model'"'"', revision='"'"'d9a0339cbf2d19f67809c1e8c575f942a99718d5'"'"', filename='"'"'training-states/cup/pi0-clean40-s7500/training_state/optimizer_state.safetensors'"'"', cache_dir='"'"'/data/Data14TB/cache/huggingface'"'"', local_dir='"'"'/data/Data14TB/01Proj/every-embodied-amd/restored-training-state'"'"'))'
大文件恢复与完整校验关闭；本轮只核对历史归档证据和损坏夹具。


## 5. 恢复训练还需要什么

除优化器外，还要保留对应步数的模型、训练配置、调度器与随机状态，并确认数据顺序。此案例只归档旧优化器，模型权重保留；不能把“优化器文件校验成功”直接当作“整套训练恢复已验证”。18号和四篇排障课分别说明模型身份、加载、随机状态与动作时钟。
